In [ ]:
import torch
import torch.nn as nn

# Problem 3: 实现 RMSNorm (1 point)
class RMSNorm(torch.nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5, device=None, dtype=None):
        super().__init__()
        '''
        d_model: int  模型隐藏层的特征维度
        eps: float = 1e-5  数值稳定性极小值，防止分母为 0
        device: torch.device | None = None  参数存放设备
        dtype: torch.dtype | None = None  参数数据类型
        '''
        self.d_model = d_model
        # 可学习缩放参数 gamma，初始化为全 1
        self.g = nn.Parameter(torch.ones(d_model, device=device, dtype=dtype)) # (d_model, )
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        输入形状: [..., d_model] 支持任意前置维度（如 batch_size, seq_len），最后一维为特征维度
        输出形状: [..., d_model] 与输入形状完全一致
        '''
        # 记录原始精度，计算时提升到 float32 保证稳定，输出时还原精度
        in_dtype = x.dtype
        x = x.to(torch.float32)

        # 核心计算：求均方根 -> 归一化 -> 乘可学习缩放系数
        # (..., d_model) -> (..., 1)
        rms = (x.square().sum(dim=-1, keepdim=True) / self.d_model + self.eps) ** 0.5 # (..., 1)
        result = x / rms * self.g # (..., d_model) / (..., 1) * (d_model) -> (..., d_model)

        return result.to(in_dtype)

In [ ]:
# 构造 3 行 3 列的输入张量：3 个样本，每个样本 3 维特征
x = torch.tensor([
    [1.0, 2.0, 3.0],
    [0.0, -1.0, 2.0],
    [40.0, 80.0, 120.0]
])

# 初始化 RMSNorm，特征维度为 3
norm = RMSNorm(d_model=3)
output = norm(x)

print("输入张量：")
print(x)
print("\nRMSNorm 输出张量：")
print(output)

In [ ]:
import torch
from einops import rearrange

# Problem 4: 实现 RoPE (2 points)
class RotaryPositionalEmbedding(torch.nn.Module):
    def __init__(self, theta: float, d_k: int, max_seq_len: int, device=None):
        super().__init__()
        '''
        theta: float  RoPE 的 Θ 值
        d_k: int  待编码的向量维度（一般指多头注意力层里 query 和 key 向量的维度）
        max_seq_len: int  输入的最大序列长度
        device: torch.device | None = None  存储 buffer 的设备
        '''
        
        assert d_k % 2 == 0, "RoPE 要求向量维度 d_k 必须是偶数"

        # 预计算频率与角度表
        # 旋转角度 angle[i][k] = i/(theta**(2k-2)/d), i 为 token 位置下标，k 为词向量内的元素两两分组的组号
        positions = torch.arange(max_seq_len, device=device) # 旋转角度的分子 i: [0, 1, 2, 3, ..., max_seq_len - 1]
        dim_indices = torch.arange(0, d_k, 2, device=device) # 2k‑2: [0, 2, 4, 6..., d_k - 2]; k: [1, 2, 3, ..., d_k//2]

        freqs = theta ** -(dim_indices / d_k) # 频率，也即旋转角度的分母 theta**(2k‑2)/d

        # 逐位置乘构造角度表（依赖广播机制）： (max_seq_len, 1) * (1, d_k//2) -> (max_seq_len, d_k//2)
        angle = positions[:, None] * freqs[None, :]

        cos_table = torch.cos(angle)
        sin_table = torch.sin(angle)

        # 把预先计算好的 cos 表和 sin 表注册为 buffer 缓存起来，方便以后重复使用
        self.register_buffer("cos_table", cos_table)
        self.register_buffer("sin_table", sin_table)

    def forward(self, x: torch.Tensor, token_positions: torch.Tensor) -> torch.Tensor:
        '''
        x: (..., seq_len, d_k) 输入张量
        token_positions: (..., seq_len) 存储每个 token 的位置下标，支持位置非连续的 token 序列： (1, 3, 5, 7...)
        return: (..., seq_len, d_k) 旋转位置编码后的张量
        '''
        cos = self.cos_table[token_positions] # (..., seq_len) -> (..., seq_len, d_k//2): 使用 token 位置下标 i 对 cos 表的第 0 维取元素，取得第 i 行的 cos 序列
        sin = self.sin_table[token_positions] # (..., seq_len) -> (..., seq_len, d_k//2): 使用 token 位置下标 i 对 sin 表的第 0 维取元素，取得第 i 行的 sin 序列

        # 两两分组后执行旋转
        # 把最后一维 d_k 按「每 2 个元素为一组」切开，每组 [x_even, x_odd]，全部组的偶数元素汇总为 x1，奇数元素汇总为 x2
        x1, x2 = rearrange(x, "... (half_d xy) -> ... half_d xy", xy=2).unbind(dim=-1)  # (..., seq_len, d_k) -> (..., seq_len, d_k//2, 2) -> x1/x2: (..., seq_len, d_k//2)

        # 分别对奇数位置的元素和偶数位置的元素执行不同的旋转操作
        x1_rot = x1 * cos - x2 * sin # (..., seq_len, d_k//2)
        x2_rot = x1 * sin + x2 * cos # (..., seq_len, d_k//2)

        # 拼接回原始形状
        # x_rot = torch.concat([x1_rot, x2_rot], dim=-1) # 用 concat 导致最后一维元素的顺序会变为 (x0, x2, x1, x3)，不符合原始的 (x0, x1, x2, x3)
        x_rot = torch.stack([x1_rot, x2_rot], dim=-1) # (..., seq_len, d_k//2) -> (..., seq_len, d_k//2, 2)
        x_rot = rearrange(x_rot, "... half_d xy -> ... (half_d xy)") # (..., seq_len, d_k//2, 2) -> (..., seq_len, d_k)
        return x_rot

In [ ]:
# 构造输入张量：2 个样本，每个样本 4 个 token，每个 token 特征维度为 4（d_k 必须为偶数）
# shape: [batch_size=2, seq_len=4, d_k=4]
x = torch.tensor([
    [[1.0, 2.0, 3.0, 4.0],
     [0.0, -1.0, 2.0, 5.0],
     [40.0, 80.0, 120.0, 160.0],
     [2.0, 3.0, 5.0, 7.0]],

    [[10.0, 20.0, 30.0, 40.0],
     [-1.0, -2.0, -3.0, -4.0],
     [5.0, 6.0, 7.0, 8.0],
     [0.0, 0.0, 0.0, 0.0]]
])

# 每个 token 对应的绝对位置下标，shape: [batch_size=2, seq_len=4]
# 这里两个样本的位置都是 0,1,2,3
token_pos = torch.tensor([
    [0, 1, 2, 3],
    [0, 1, 2, 3]
])

# 初始化 RoPE：theta=10000.0，特征维度 d_k=4，支持最大序列长度 max_seq_len=1024
rope = RotaryPositionalEmbedding(theta=10000.0, d_k=4, max_seq_len=1024)
output = rope(x, token_pos)

print("输入张量 x：")
print(x)
print("\nRoPE 输出张量：")
print(output)
print(f"\n输入形状: {x.shape}, 输出形状: {output.shape}")